In [ ]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy import sparse, special, optimize
import numpy as np
from typing import Literal
import anndata as ad
import statsmodels.api as sm

ad.settings.allow_write_nullable_strings = True
sc.settings.figdir = "./results/figures"
#sc.settings.verbosity = 3




Loading the h5ad file from the P03_qc_filtering step.
We will have the following structure:
  - `adata.layer['counts']` to preserve the raw counts to be later used in further
  steps downstream. 
  - `adata.layer['lop1p_norm']` to have the normalized total counts of cells and the
  log1p on that, it is easier # !for visualization ONLY.
  - Tag Only (`adata.var['highly_variable']`), highly variable genes using analytic Pearson residuals HVG on adata.X raw
  - Estimate_global_theta ($\theta = {1/{\alpha}}$) instead of standard 100 for the next step of Pearson residual normalization 
  - Apply analytic Pearson residual normalization on adata.X raw (all genes)
  - Save the modified adata into h5ad

**We are solving for the Global Overdispersion $\alpha$ (where $\theta = 1/\alpha$) :**

_Variance is strictly positive and its error scales with its magnitude. This is the definition of a Gamma Distribution_.

  By using a **Gamma GLM**, we respect the physics of the "Universe Plot". The "Weight Matrix W" is the core of the **IWLS (Iteratively Reweighted Least Squares)** algorithm, which `statsmodels` will handle dynamically for us.

### The Physics of the Solution

  Using the equation:

  $$(\text{Variance} - \text{Mean}) = \alpha \times (\text{Mean}^2)$$

  - **Y (Response):** Excess Variance. (Gamma Distributed).
      
  - **X (Predictor):** Mean Squared.
      
  - **Link Function:** Identity ($Y = \alpha X$).
      
  - **Family:** Gamma.
    
  Instead of using a standard 100 value, we will calculate it by using the GLM,
  IWLS implementation on the raw alphas and means of each gene. and after the
  regression, find a true theta , with the design matrix of the ~13000 mean
  expressions and Weights (W) (Derived from the Degrees of Freedom(how many cells contributed to each dot)). 
  - The Distribution: Gamma (instead of Negative Binomial).
  - The Link Function: Log (to ensure the result is always positive).
'''

In [ ]:
def load_evidence(h5ad_path):

    # Loads the QC-cleaned artifact from P03 file
    print(f"   -> Loading QC artifact from: {h5ad_path}")
    adata = sc.read_h5ad(h5ad_path)
    print(f"   -> Loaded dimensions: {adata.n_obs} cells x {adata.n_vars} genes")
    return adata

    


In [ ]:
def data_layering(adata):
    '''
    create the adata.layer['counts'], adata.layer['lop1p_norm'],and keeping the
    adata.X clean and raw from the P03 file
    '''
    
    adata.layers['counts'] = adata.X.copy()
    adata_temp = adata.copy()
    sc.pp.normalize_total(adata_temp, target_sum = 1e4)
    sc.pp.log1p(adata_temp)
    adata.layers['log1p_norm'] = adata_temp.X.copy()
    del adata_temp
    return adata


In [ ]:
def npr_and_hvg(adata):
    
    # we will tag the genes as HVG or not
    # Taking theta Value to be 100.0 as from experiments in the 
    sc.experimental.pp.highly_variable_genes(adata,theta=100.0,n_top_genes=3000,flavor='pearson_residuals',subset=False)
    # Pearson residuals calculation
    sc.experimental.pp.normalize_pearson_residuals(adata, theta = 100.0)
    
    return adata




In [ ]:
if __name__ == "__main__":
    h5ad_path = "/Users/qgem/GitHub/PBMC3k-reproducible/data/objects/pbmc3k_qc.h5ad"
    adata = load_evidence(h5ad_path)
    adata = data_layering(adata)
    adata = npr_and_hvg(adata)
    
b = adata.var
c = np.array(adata.var.columns)

In [11]:
adata = load_evidence('/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/adata_project_macro_leiden_2_Terminal_State.h5ad')
adata_t = load_evidence('/Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/adata_train_macro_leiden_2_Terminal_State.h5ad')
df_p = adata.obs
df_t = adata_t.obs
df = pd.concat([df_p,df_t])
duplicate_count = df.index.duplicated().sum()
if duplicate_count>0:
    print(f" BARCODE COLLISION DETECTED.")
    

   -> Loading QC artifact from: /Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/adata_project_macro_leiden_2_Terminal_State.h5ad
   -> Loaded dimensions: 164 cells x 13456 genes
   -> Loading QC artifact from: /Users/qgem/GitHub/PBMC3k-reproducible/notebooks/results/adata_train_macro_leiden_2_Terminal_State.h5ad
   -> Loaded dimensions: 175 cells x 13456 genes
